### PDF'i Parçalara Bölüp Gömme Vektörlerini Oluşturun

In [1]:
import os
from dotenv import load_dotenv
from pypdf import PdfReader
import chromadb
from google import genai

load_dotenv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje\notepads\.env")

# PDF'i metne cevirin
reader = PdfReader(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje\rag_docs\pjm_manual13.pdf")
full_text = "\n".join(page.extract_text() for page in reader.pages)

# Basit, sabit-boyutlu parcalama (kucuk dokuman seti icin yeterli)
chunk_size = 1000
overlap = 200
chunks = []
start = 0
while start < len(full_text):
    chunk = full_text[start:start + chunk_size]
    dot_ratio = chunk.count('.') / max(len(chunk), 1)
    if dot_ratio < 0.15:
        chunks.append(chunk)
    start += chunk_size - overlap

print(f"Toplam parca sayisi: {len(chunks)}")

Toplam parca sayisi: 568


### ChromaDB'ye Kaydedin (Sunucu Gerektirmez)

In [2]:
chroma_client = chromadb.PersistentClient(path=r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje\rag_docs\chroma_store")

# Eski, filtresiz koleksiyonu tamamen silip sifirdan olusturuyoruz
try:
    chroma_client.delete_collection(name="pjm_manual13")
except Exception:
    pass
collection = chroma_client.create_collection(name="pjm_manual13")

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
for i, chunk in enumerate(chunks):
    embedding = gemini_client.models.embed_content(
        model="gemini-embedding-001", contents=chunk
    ).embeddings[0].values
    collection.add(ids=[f"chunk_{i}"], embeddings=[embedding], documents=[chunk])

print("Kaydedilen parca sayisi:", collection.count())

Kaydedilen parca sayisi: 568


### Sorgulama Fonksiyonunu Yazın (LangSmith İzlemeli)

In [3]:
### Sorgulama Fonksiyonunu Yazın (LangSmith İzlemeli)

from langsmith import traceable

@traceable(name="pjm_manual13_retrieval")
def retrieve_context(query, k=3):
    query_embedding = gemini_client.models.embed_content(
        model="gemini-embedding-001", contents=query
    ).embeddings[0].values
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    return "\n---\n".join(results["documents"][0])


def build_rag_query(forecast_mw, available_capacity_mw, time_period):
    margin_pct = (available_capacity_mw - forecast_mw) / available_capacity_mw
    if margin_pct >= 0.20:
        query = f"PJM reserve requirements control zone normal operating margin {time_period}"
    else:
        query = f"grid capacity shortfall reserve activation {time_period}"
    return query